# GFS 00Z surface temperature and snow mask GIF

Download selected GFS 0.25 degree fields from the March 25, 2022 00Z run for forecast hours 0-23 over 109W-104W and 37N-41N. The notebook crops the selected GRIB messages, derives a snow mask, writes a compact NetCDF, and saves a 24-frame GIF of surface temperature and snow mask.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import pandas as pd
import requests
import xarray as xr

if importlib.util.find_spec("cfgrib") is None:
    raise ImportError(
        "This notebook needs cfgrib to open GFS GRIB2 files. "
        f"Current Python is {sys.executable}. "
        "Switch the notebook kernel to 'Python (goes_downloading)' and rerun."
    )

import cfgrib  # noqa: F401  # registers the xarray cfgrib backend

if "cfgrib" not in xr.backends.list_engines():
    raise RuntimeError(
        "cfgrib is installed but xarray did not register it as an engine. "
        f"Current Python is {sys.executable}. "
        "Restart the kernel or switch to 'Python (goes_downloading)'."
    )

print(f"Python: {sys.executable}")
print(f"xarray engines: {sorted(xr.backends.list_engines())}")

In [ ]:
# ---- Configuration ----
DATE = "20220325"
CYCLE = "00"
FORECAST_HOURS = list(range(24))
PRODUCT_TEMPLATE = "pgrb2.0p25.f{forecast_hour:03d}"
OVERWRITE = False

# Requested domain: 104 to 109W and 37 to 41N.
LON_MIN = -109.0
LON_MAX = -104.0
LAT_MIN = 37.0
LAT_MAX = 41.0

# GFS GRIB files use longitudes on 0-360.
GFS_LON_MIN = LON_MIN % 360
GFS_LON_MAX = LON_MAX % 360

OUT_DIR = Path("../data_download/gfs_surface_20220325").resolve()
FRAME_DIR = OUT_DIR / "gfs_20220325_00z_f000_f023_gif_frames"
NETCDF_PATH = OUT_DIR / "gfs_20220325_00z_f000_f023_surface_temperature_snow_mask_109W_104W_37N_41N.nc"
GIF_PATH = OUT_DIR / "gfs_20220325_00z_f000_f023_surface_temperature_snow_mask.gif"

FIELDS = {
    ("TMP", "surface"),    # land/skin surface temperature, K
    ("WEASD", "surface"),  # snow water equivalent, kg m**-2
    ("SNOD", "surface"),   # snow depth, m
    ("CSNOW", "surface"),  # categorical snow
    ("LAND", "surface"),   # land-sea mask
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT_DIR}")
print(f"NetCDF output: {NETCDF_PATH}")
print(f"GIF output: {GIF_PATH}")

In [ ]:
def gfs_urls(date, cycle, forecast_hour):
    product = PRODUCT_TEMPLATE.format(forecast_hour=forecast_hour)
    base = f"https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.{date}/{cycle}/atmos"
    stem = f"gfs.t{cycle}z.{product}"
    return f"{base}/{stem}", f"{base}/{stem}.idx"


def output_paths(date, cycle, forecast_hour, out_dir=OUT_DIR):
    grib_path = out_dir / (
        f"gfs_{date}_{cycle}z_f{forecast_hour:03d}_surface_snow_tmp_selected.grib2"
    )
    idx_path = grib_path.with_suffix(".idx.txt")
    return grib_path, idx_path


def read_idx(idx_url):
    response = requests.get(idx_url, timeout=60)
    response.raise_for_status()
    lines = response.text.splitlines()

    records = []
    for i, line in enumerate(lines):
        parts = line.split(":")
        if len(parts) < 6:
            continue

        start = int(parts[1])
        end = int(lines[i + 1].split(":")[1]) - 1 if i + 1 < len(lines) else None
        records.append(
            {
                "record": int(parts[0]),
                "start": start,
                "end": end,
                "var": parts[3],
                "level": parts[4],
                "forecast": parts[5],
                "line": line,
            }
        )
    return records


def selected_records(records, fields=FIELDS):
    selected = []
    found = set()
    for record in records:
        key = (record["var"], record["level"])
        if key not in fields or key in found:
            continue
        # Forecast-hour files also contain averaged CSNOW records; keep the first
        # instantaneous record listed in the GFS index.
        selected.append(record)
        found.add(key)

    missing = fields - found
    if missing:
        raise ValueError(f"Missing requested GFS fields: {sorted(missing)}")
    return selected


def download_selected_grib(date, cycle, forecast_hour, overwrite=OVERWRITE):
    grib_path, idx_path = output_paths(date, cycle, forecast_hour)
    if grib_path.exists() and grib_path.stat().st_size > 0 and not overwrite:
        print(f"f{forecast_hour:03d}: using existing {grib_path.name}")
        return grib_path

    grib_url, idx_url = gfs_urls(date, cycle, forecast_hour)
    records = selected_records(read_idx(idx_url))
    idx_path.write_text("\n".join(r["line"] for r in records) + "\n")

    with grib_path.open("wb") as out_file:
        for record in records:
            byte_range = (
                f"bytes={record['start']}-{record['end']}"
                if record["end"] is not None
                else f"bytes={record['start']}-"
            )
            response = requests.get(
                grib_url,
                headers={"Range": byte_range},
                timeout=120,
            )
            response.raise_for_status()
            out_file.write(response.content)

    size_mb = grib_path.stat().st_size / 1024**2
    print(f"f{forecast_hour:03d}: downloaded {grib_path.name} ({size_mb:.2f} MB)")
    return grib_path

In [ ]:
grib_paths = [
    download_selected_grib(DATE, CYCLE, forecast_hour)
    for forecast_hour in FORECAST_HOURS
]

download_summary = pd.DataFrame(
    {
        "forecast_hour": FORECAST_HOURS,
        "grib_path": [str(path) for path in grib_paths],
        "size_mb": [path.stat().st_size / 1024**2 for path in grib_paths],
    }
)
download_summary

In [ ]:
def open_and_subset_grib(path, forecast_hour):
    ds = xr.open_dataset(
        path,
        engine="cfgrib",
        backend_kwargs={
            "indexpath": "",
            "filter_by_keys": {"typeOfLevel": "surface"},
        },
    )
    analysis_time = pd.Timestamp(ds["time"].item())
    valid_time = pd.Timestamp(ds["valid_time"].item())

    ds = ds.sel(
        latitude=slice(LAT_MAX, LAT_MIN),
        longitude=slice(GFS_LON_MIN, GFS_LON_MAX),
    ).copy()
    ds = ds.drop_vars(["time", "valid_time"], errors="ignore")
    ds = ds.assign_coords(longitude=((ds.longitude + 180) % 360) - 180)
    ds = ds.sortby("longitude")
    ds = ds.expand_dims(valid_time=[valid_time])
    ds = ds.assign_coords(
        cycle=("valid_time", [CYCLE]),
        forecast_hour=("valid_time", [forecast_hour]),
        analysis_time=("valid_time", [analysis_time]),
    )

    ds["surface_temperature_c"] = ds["t"] - 273.15
    ds["surface_temperature_c"].attrs.update(
        long_name="GFS surface temperature",
        units="degC",
    )

    ds["snow_mask"] = (ds["sdwe"] > 0) | (ds["sde"] > 0) | (ds["csnow"] > 0)
    ds["snow_mask"].attrs.update(
        long_name="Derived GFS snow mask",
        description="True where WEASD > 0, SNOD > 0, or CSNOW > 0",
        units="1",
    )
    return ds


domain_ds = xr.concat(
    [open_and_subset_grib(path, forecast_hour) for path, forecast_hour in zip(grib_paths, FORECAST_HOURS)],
    dim="valid_time",
    coords="minimal",
    compat="override",
)

domain_ds.attrs.update(
    title="GFS 00Z surface temperature and snow mask 24-hour forecast over Colorado domain",
    source="NOAA GFS gfs.t00z.pgrb2.0p25.f000-f023 forecast fields",
    date=DATE,
    cycle=f"{CYCLE}Z",
    forecast_hours=f"{FORECAST_HOURS[0]}-{FORECAST_HOURS[-1]}",
    domain=f"{abs(LON_MIN):.1f}W-{abs(LON_MAX):.1f}W, {LAT_MIN:.1f}N-{LAT_MAX:.1f}N",
    snow_mask_logic="WEASD > 0 or SNOD > 0 or CSNOW > 0",
)

domain_ds.to_netcdf(NETCDF_PATH)

print(f"Saved {NETCDF_PATH}")
print(f"Dimensions: {dict(domain_ds.sizes)}")
print(f"Variables: {list(domain_ds.data_vars)}")
print(f"Valid time range: {pd.Timestamp(domain_ds.valid_time.min().item())} to {pd.Timestamp(domain_ds.valid_time.max().item())}")
print(f"Latitude range: {float(domain_ds.latitude.min()):.2f} to {float(domain_ds.latitude.max()):.2f}")
print(f"Longitude range: {float(domain_ds.longitude.min()):.2f} to {float(domain_ds.longitude.max()):.2f}")

domain_ds

In [ ]:
temp_min = float(domain_ds["surface_temperature_c"].min())
temp_max = float(domain_ds["surface_temperature_c"].max())
frame_paths = []

for valid_time in domain_ds.valid_time.values:
    ds_time = domain_ds.sel(valid_time=valid_time)
    forecast_hour = int(ds_time["forecast_hour"].item())
    timestamp = pd.Timestamp(valid_time).strftime("%Y-%m-%d %HZ")
    frame_path = FRAME_DIR / f"frame_f{forecast_hour:03d}.png"

    fig, axes = plt.subplots(ncols=2, figsize=(10, 4.5), constrained_layout=True)

    ds_time["surface_temperature_c"].plot(
        ax=axes[0],
        x="longitude",
        y="latitude",
        cmap="coolwarm",
        vmin=temp_min,
        vmax=temp_max,
        cbar_kwargs={"label": "degC"},
    )
    axes[0].set_title("Surface temperature")

    ds_time["snow_mask"].astype("int8").plot(
        ax=axes[1],
        x="longitude",
        y="latitude",
        cmap="Greys",
        vmin=0,
        vmax=1,
        cbar_kwargs={"label": "snow mask"},
    )
    axes[1].set_title("Snow mask")

    for ax in axes:
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    fig.suptitle(f"GFS {DATE} {CYCLE}Z run | f{forecast_hour:03d} | valid {timestamp}")
    fig.savefig(frame_path, dpi=130)
    plt.close(fig)
    frame_paths.append(frame_path)

images = [imageio.imread(frame_path) for frame_path in frame_paths]
imageio.mimsave(GIF_PATH, images, duration=0.45, loop=0)

print(f"Saved {GIF_PATH}")
print(f"Frames: {len(frame_paths)}")
GIF_PATH

## Five-panel GOES/GFS GIF

Build the 5-panel 5-minute GIF from 16Z to 00Z with RGB composite, GOES BCM cloud mask, RGB cloud mask, GFS snow mask, and GFS surface temperature. The RGB mask uses the dynamic 10C GFS surface-temperature threshold bins, and the hourly GFS fields are reused for each 5-minute GOES frame in the same UTC hour.


In [ ]:
!python build_20220325_five_panel_gif.py
